# **2026-2 · TÉCNICAS AVANZADAS DE DATA MINING Y SISTEMAS INTELIGENTES (INF659)**
## **Examen Parcial — (3 horas)**

### Descripción de la tarea
El objetivo es predecir si un paciente presenta rastros de neumonía asociada a COVID-19 a partir de **tomografías computarizadas (TC) de pulmón en 3D**. El dataset contiene volúmenes de TC anonimizados con hallazgos anormales (COVID) y normales.

**Dataset (fuente):** https://www.medrxiv.org/content/10.1101/2020.05.20.20100362v1
Se usa una porción de **100 volúmenes por clase** (normal / anormal).

>  **diseccione, intervenga, diagnostique y defienda** sus decisiones.

## Reglas del examen

- **Tiempo:** 3 horas, modalidad para casa. Trabajo **individual**.
- **Uso de IA:** está **permitido y se espera** que use asistentes de IA / agentes de código. No es trampa. Pero lea con atención cómo se evalúa.
- **Qué se evalúa:** la comprensión, no la ejecución. Casi todas las preguntas dependen de **los resultados de SU propia corrida** (sus curvas, sus tablas de ablación, sus mapas de calor), que un asistente de IA no puede conocer por usted.
- **Video de sustentación (OBLIGATORIO):** suba un video de **máximo 10 minutos** explicando su solución: sus decisiones de diseño, sus resultados de ablación, su análisis de transfer learning, la interpretación de sus Grad-CAM y su diagnóstico. **Los puntos de cada parte se acreditan únicamente si usted puede explicar ese trabajo en el video.** Si no sabe explicar por qué algo está en su código, no recibe el puntaje.
- **Cuaderno de decisiones:** en cada parte hay un espacio para registrar *por qué* tomó cada decisión. Un asistente ejecuta; su justificación demuestra que usted dirigió.
- **Honestidad:** reporte también **lo que NO funcionó**. Un resultado negativo bien analizado vale más que un número inflado.
- **Reproducibilidad:** no modifique las semillas fijadas. Si reduce resolución/épocas para ganar tiempo, decláreo y justifíquelo.
- **Entrega:** este notebook ejecutado + el enlace al video. Total: **20 puntos**.

### Declaración de uso de IA *(obligatoria — complete antes de entregar)*

Indique qué herramientas de IA usó y para qué (p. ej. generación de código, depuración, redacción). Sea específico.

- Herramienta(s):
- Para qué la usé:
- Qué partes resolví/decidí yo directamente:

### Enlace al video de sustentación (máx. 10 min)

**URL:** `________________________`

In [7]:
import tensorflow as tf

from tensorflow.keras.layers import Input # Importe aquí el resto de capas que necesite
from tensorflow.keras import Model
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from tensorflow.keras import backend as K

tf.random.set_seed(0) # No modificar esta línea
import numpy as np
np.random.seed(0)      # No modificar esta línea

In [8]:
import os
import nibabel as nib
from scipy import ndimage
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
%matplotlib inline

---
## Carga y preprocesamiento de datos *(provisto — no es lo que se evalúa)*

Las TC están en formato Nifti (`.nii`), con intensidades en unidades Hounsfield (HU), rango [-1024, 2000+]. Se recorta a [-1000, 400] y se normaliza a [0, 1]; cada volumen se redimensiona a (128, 128, 64).

In [9]:
!git clone https://github.com/iapucp/CT-Scans-100-samples.git
%mv CT-Scans-100-samples/CT-Normal /content/CT-Normal
%mv CT-Scans-100-samples/CT-Abnormal /content/CT-Abnormal
%rm -rf CT-Scans-100-samples/

Cloning into 'CT-Scans-100-samples'...
remote: Enumerating objects: 204, done.
remote: Total 204 (delta 0), reused 0 (delta 0), pack-reused 204 (from 1)
Receiving objects: 100% (204/204), 1.96 GiB | 23.94 MiB/s, done.
Updating files: 100% (200/200), done.


In [10]:
normal_scan_paths   = ["/content/CT-Normal/"   + x for x in os.listdir("CT-Normal")]
abnormal_scan_paths = ["/content/CT-Abnormal/" + x for x in os.listdir("CT-Abnormal")]
print("Normal:",   len(normal_scan_paths))
print("Abnormal:", len(abnormal_scan_paths))

Normal: 101
Abnormal: 101


In [11]:
def normalize_scan(scan, min=-1000, max=400):
  scan[scan < min] = min
  scan[scan > max] = max
  scan = (scan - min) / (max - min)
  return scan.astype("float32")

def resize_scan(scan, w, h, d):
  cw, ch, cd = scan.shape[0], scan.shape[1], scan.shape[-1]
  scan = ndimage.rotate(scan, 90, reshape=False)
  scan = ndimage.zoom(scan, (w/cw, h/ch, d/cd), order=1)
  return scan

def preprocess_scan(path, w, h, d):
  scan = nib.load(path).get_fdata()
  return normalize_scan(resize_scan(scan, w, h, d))

width, height, depth = 128, 128, 64

In [13]:
%%time
# Filtramos las listas de rutas para asegurarnos de que solo procesamos archivos Nifti (.nii o .nii.gz)
normal_paths_filtered = [p for p in normal_scan_paths if p.endswith('.nii') or p.endswith('.nii.gz')]
abnormal_paths_filtered = [p for p in abnormal_scan_paths if p.endswith('.nii') or p.endswith('.nii.gz')]

normal_scans   = np.array([preprocess_scan(p, width, height, depth) for p in normal_paths_filtered])
abnormal_scans = np.array([preprocess_scan(p, width, height, depth) for p in abnormal_paths_filtered])

normal_labels   = np.zeros(normal_scans.shape[0],   dtype=int)
abnormal_labels = np.ones(abnormal_scans.shape[0],  dtype=int)

scans  = np.concatenate([normal_scans,  abnormal_scans],  axis=0)
labels = np.concatenate([normal_labels, abnormal_labels], axis=0)
del normal_scans, abnormal_scans, normal_labels, abnormal_labels

CPU times: user 7min 19s, sys: 16.5 s, total: 7min 35s
Wall time: 7min 43s


In [15]:
if 'scans' in locals() or 'scans' in globals():
    x_train, x_test, y_train, y_test = train_test_split(
        scans, labels, test_size=0.15, random_state=0, stratify=labels)
    del scans
    print("Partición creada de manera exitosa.")
else:
    print("La partición ya fue creada y la variable 'scans' ya se liberó de la memoria.")

print("Dimensiones del dataset:")
print("x_train:", x_train.shape, "y_train:", y_train.shape)
print("x_test:", x_test.shape, "y_test:", y_test.shape)

La partición ya fue creada y la variable 'scans' ya se liberó de la memoria.
Dimensiones del dataset:
x_train: (170, 128, 128, 64) y_train: (170,)
x_test: (30, 128, 128, 64) y_test: (30,)


In [16]:
from random import choice, seed
seed(0)

@tf.function
def rotate(scan):
  def scipy_rotate(scan):
    angle = choice([-20, -10, -5, 5, 10, 20])
    scan = ndimage.rotate(scan, angle, reshape=False)
    scan[scan < 0] = 0; scan[scan > 1] = 1
    return scan
  return tf.numpy_function(scipy_rotate, [scan], tf.float32)

def train_aug(scan, label):
  scan = rotate(scan)
  return tf.expand_dims(scan, axis=3), label

def test_aug(scan, label):
  return tf.expand_dims(scan, axis=3), label

batch_size = 2 # (preste atención a este valor — se le preguntará por él)
ds_train = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(len(x_train)).map(train_aug).batch(batch_size).prefetch(2)
ds_test  = tf.data.Dataset.from_tensor_slices((x_test,  y_test )).map(test_aug).batch(batch_size).prefetch(2)

In [17]:
def show_results(log):
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,4))
  ax1.plot(log.history['loss'], label='train'); ax1.plot(log.history['val_loss'], label='val')
  ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend()
  ax2.plot(log.history['acc'], label='train'); ax2.plot(log.history['val_acc'], label='val')
  ax2.set_xlabel('epoch'); ax2.set_ylabel('accuracy'); ax2.legend()
  plt.show()

---
## P1 — Línea base e instrumentación **(3 puntos)**

Construya un modelo **base** sencillo (unas pocas capas `Conv3D` + `pooling` + una cabeza densa → `sigmoid`).

**Antes de entrenar, responda (predicción):**
1. Con ~170 volúmenes de entrenamiento y `batch_size=2`, ¿espera sobreajuste? Describa **cómo predice** que se verá la brecha entre `train` y `val` (loss y accuracy).

Luego entrene, muestre las curvas con `show_results`, e **instrumente** el modelo:
2. Reporte el **número total de parámetros** y cuántos aporta cada bloque (puede usar `model.summary()`).
3. Compare su **predicción** del punto 1 con lo que **realmente** ocurrió. ¿Acertó? Si no, ¿por qué cree que se equivocó?

In [ ]:
K.clear_session()
inputs = Input(shape=(width, height, depth, 1))

# TODO (P1): construya aquí su modelo base (pocas capas convolucionales + cabeza densa)
# x = ...

outputs = Dense(units=1, activation="sigmoid")(x)
model = Model(inputs=inputs, outputs=outputs)
model.summary()

In [ ]:
initial_learning_rate = 0.0001
lr_schedule = ExponentialDecay(initial_learning_rate, decay_steps=..., decay_rate=...) # TODO

model.compile(optimizer=Adam(learning_rate=lr_schedule),
              loss="binary_crossentropy", metrics=["acc"])

checkpoint     = ModelCheckpoint("base-{epoch:02d}-{val_acc:.3f}.weights.h5",
                                 monitor="val_acc", mode="max", save_best_only=True, save_weights_only=True)
early_stopping = EarlyStopping(monitor="val_acc", patience=..., mode="max") # TODO
callbacks = [checkpoint, early_stopping]

log = model.fit(ds_train, epochs=..., callbacks=callbacks, validation_data=ds_test) # TODO
show_results(log)

**P1 — Sus respuestas / cuaderno de decisiones**

- Predicción (antes de entrenar) →
- Parámetros totales y por bloque →
- Predicción vs realidad (¿acertó? ¿por qué?) →
- Decisiones de diseño del modelo base →

---
## P2 — Intervención arquitectónica y ablación **(6 puntos)**

Compare **tres variantes** de arquitectura sobre el mismo problema. Para **cada una**, escriba una **hipótesis antes de correr**, entrénela y registre sus números en la tabla.

- **(a) Base** — su modelo de P1 con cabeza densa.
- **(b) Conexiones residuales** — introduzca bloques residuales 3D (estilo ResNet): `salida = F(x) + x`. Explique qué resuelven aquí.
- **(c) Fully-convolutional con Global Average Pooling** — elimine las densas finales y use `GlobalAveragePooling3D` antes de la salida (estilo *Network in Network* / GoogLeNet).

Lo que se evalúa es la **tabla completa con SUS resultados** y su **explicación de los deltas**, no que cada variante alcance el mejor accuracy.

**Recuerde:** no es lo mismo una capa convolucional con 1024 filtros que con 128. Controle los demás factores para que la comparación sea justa.

In [ ]:
def residual_block_3d(x, filters):
    # TODO (P2b): implemente un bloque residual 3D.
    # Pista: dos Conv3D (con BatchNorm/activación) y un atajo que sume la entrada.
    #        cuide que las dimensiones (canales/espaciales) permitan la suma -> proyección 1x1x1 si hace falta.
    return x

K.clear_session()
inputs = Input(shape=(width, height, depth, 1))
# x = ... usando residual_block_3d(...)
outputs = Dense(1, activation="sigmoid")(x)
model_res = Model(inputs, outputs)
model_res.summary()
# ... compile + fit (reutilice el patrón de P1, cambiando el nombre del checkpoint)

In [ ]:
from tensorflow.keras.layers import GlobalAveragePooling3D

K.clear_session()
inputs = Input(shape=(width, height, depth, 1))
# TODO (P2c): bloques convolucionales -> GlobalAveragePooling3D -> Dense(1, 'sigmoid')
outputs = Dense(1, activation="sigmoid")(x)
model_gap = Model(inputs, outputs)
model_gap.summary()
# ... compile + fit

**P2 — Tabla de ablación (complete con SUS resultados)**

| Variante | #Parámetros | Val. accuracy | Val. F1 | Épocas (early stop) | Observación clave |
|---|---|---|---|---|---|
| (a) Base (densa) |  |  |  |  |  |
| (b) Residual 3D  |  |  |  |  |  |
| (c) GAP (fully-conv) |  |  |  |  |  |

**Análisis (cuaderno de decisiones):**
- Hipótesis previa de cada variante →
- ¿Por qué cambian los resultados entre (a), (b) y (c)? Relacione con lo visto en clase (flujo del gradiente en residuales, rol de GAP, número de parámetros) →
- ¿Cuál elegiría y por qué? →

---
## P3 — Transfer learning 2D→3D **(4 puntos)**

Los backbones pre-entrenados (ResNet/EfficientNet) son **2D y de ImageNet**; aquí los datos son **3D y de TC médica**. Elija **una** estrategia de adaptación y compárela contra su mejor modelo entrenado desde cero:

- **(i) Slice-wise:** aplique un backbone 2D pre-entrenado a cada corte (slice) y agregue las características (p. ej. pooling sobre la profundidad), **o**
- **(ii) Inflación 3D:** infle los pesos 2D a kernels 3D.

Para la estrategia elegida, pruebe **feature extraction** (backbone congelado) **vs fine-tuning**.

**Responda:**
1. ¿El transfer learning ayudó frente a entrenar desde cero? Muestre números y discuta el **domain gap** (imágenes naturales → TC).
2. **Trampa de BatchNorm:** con `batch_size=2`, ¿qué problema aparece con las capas de BatchNorm (propias y del backbone pre-entrenado)? ¿Cómo lo maneja?

In [ ]:
# TODO (P3): implemente su estrategia de transfer learning.
# Pistas:
#  - slice-wise: TimeDistributed(backbone_2d) sobre el eje de profundidad + pooling, o un bucle por slice.
#  - recuerde replicar a 3 canales si el backbone 2D lo exige, y ajustar el preprocesamiento del backbone.
#  - para feature extraction: base.trainable = False  (ojo con las capas BatchNorm).
#  - para fine-tuning: libere capas superiores con un learning rate mucho menor.
pass

**P3 — Sus respuestas / cuaderno de decisiones**

- Estrategia elegida y por qué →
- Feature extraction vs fine-tuning (números) →
- ¿Ayudó el transfer learning? Domain gap →
- Manejo de BatchNorm con batch_size=2 →

---
## P4 — Interpretabilidad: ¿mira el modelo los pulmones? **(3 puntos)**

Implemente **Grad-CAM** (o CAM, si usó la variante con GAP) sobre su mejor modelo y genérelo para:
- **2 volúmenes bien clasificados**, y
- **2 volúmenes mal clasificados**.

Superponga el mapa de calor sobre algunos cortes y **analice**: ¿el modelo atiende al tejido pulmonar / a las lesiones, o a **artefactos espurios** (bordes de la imagen, contorno del cuerpo, fondo)? Conecte con lo visto sobre GAP → mapas de activación de clase.

In [ ]:
def grad_cam_3d(model, volume, last_conv_layer_name, class_index=0):
    # TODO (P4): implemente Grad-CAM para Conv3D.
    # Pasos:
    #  1) construya un modelo que devuelva (activaciones de la última conv, predicción)
    #  2) con tf.GradientTape, calcule el gradiente del score respecto a esas activaciones
    #  3) promedie los gradientes por canal (pesos) y combine con las activaciones -> heatmap 3D
    #  4) ReLU + normalice el heatmap
    # retorne el heatmap 3D (redimensionado al tamaño del volumen)
    raise NotImplementedError

# TODO: visualice algunos cortes con el heatmap superpuesto para casos acertados y fallados.

**P4 — Sus respuestas**

- ¿Dónde mira el modelo en los casos acertados? →
- ¿Y en los fallados? ¿Detecta algún sesgo/artefacto espurio? →
- ¿Qué le dice esto sobre la confiabilidad clínica del modelo? →

---
## P5 — Diagnóstico y rigor experimental **(4 puntos)**

Responda con base en **sus propios resultados** y en las características del montaje (dataset pequeño, 3D, `batch_size=2`, clases balanceadas):

1. **Fiabilidad.** Su conjunto de test tiene ~30 volúmenes. ¿Qué tan confiable es el accuracy que reportó? Estime un **intervalo de confianza** o ejecute **k-fold estratificado** y reporte media ± desviación. ¿Cambia su conclusión sobre cuál es el mejor modelo?
2. **BatchNorm + batch pequeño.** Explique en detalle cómo `batch_size=2` afecta a BatchNorm durante entrenamiento e inferencia, y qué alternativa usaría.
3. **Métrica y costo clínico.** ¿Es el accuracy la métrica adecuada para un *tamizaje* de neumonía? Considere el costo de un **falso negativo** (no detectar COVID) y compare con F1 / recall / AUC. ¿Qué optimizaría y por qué?
4. **Modo de fallo.** A partir de sus Grad-CAM (P4), identifique **un** modo de fallo concreto de su mejor modelo y proponga una corrección accionable.

In [ ]:
# (Opcional) Scaffold para k-fold estratificado / estimación de variabilidad.
# TODO (P5.1): entrene su mejor arquitectura en k particiones y reporte media ± std de val_acc / F1.
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
pass

**P5 — Sus respuestas**

1. Fiabilidad (IC / k-fold, media ± std) →
2. BatchNorm + batch_size=2 →
3. Métrica y costo clínico →
4. Modo de fallo + corrección →

---
## Checklist de entrega

- [ ] Notebook ejecutado con las celdas de salida visibles (curvas, tablas, Grad-CAM).
- [ ] Tabla de ablación de P2 completa con sus números.
- [ ] Declaración de uso de IA completada.
- [ ] **Enlace al video de sustentación (≤ 10 min)** pegado arriba.
- [ ] Cuadernos de decisiones de cada parte respondidos.

**Recuerde:** cada parte se acredita solo si puede explicarla en su video. Reporte también lo que no funcionó.